# Session 12 — Apache Spark: Data Sources API
**Duration:** 80 minutes · **Level:** Introductory / Intermediate · **Language:** PySpark

**Learning outcomes:** distinguish built-in and connector-based sources; read and write CSV, JSON, and Parquet; explain JDBC and MongoDB integration; compare schema inference and explicit schemas.

## 0–10 min · Live discussion: Are you following me?
Ask: What is the difference between Spark and a database? Where does a DataFrame come from? What is lazy evaluation? What happens when Spark reads 100 files?

**Think–pair–share:** Name two data formats you have used and one limitation of each.

## Environment setup
Run the next two cells. They generate a small, reproducible retail dataset; no downloads or credentials are required.

In [ ]:
# Run once in a fresh environment if PySpark is missing:
# %pip install pyspark==3.5.6
from pyspark.sql import SparkSession, functions as F, types as T
from pathlib import Path
import tempfile, json, csv
spark = SparkSession.builder.master("local[*]").appName("SparkTeaching").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

In [ ]:
base = Path(tempfile.mkdtemp(prefix="spark_course_"))
records = [
 (1,"2026-01-05","Madrid","Books",2,15.0,"online"),
 (2,"2026-01-06","Barcelona","Electronics",1,199.0,"store"),
 (3,"2026-01-07","Madrid","Books",3,12.5,"store"),
 (4,"2026-01-08","Valencia","Home",2,35.0,"online"),
 (5,"2026-01-09","Sevilla","Electronics",1,89.0,"online"),
 (6,"2026-01-10","Barcelona","Home",4,22.0,"store"),
 (7,"2026-01-11","Madrid","Electronics",2,55.0,"online"),
 (8,"2026-01-12","Valencia","Books",1,18.0,"store"),
 (9,"2026-01-13","Sevilla","Home",3,25.0,"online"),
 (10,"2026-01-14","Madrid","Home",1,45.0,"store")
]
cols = ["order_id","order_date","city","category","quantity","unit_price","channel"]
with (base / "sales.csv").open("w",newline="",encoding="utf-8") as f:
 w=csv.writer(f);w.writerow(cols);w.writerows(records)
with (base / "sales.json").open("w",encoding="utf-8") as f:
 for r in records:f.write(json.dumps(dict(zip(cols,r)))+"\n")
print("Teaching data directory:",base)

## 10–50 min · Lecture: Core and community-driven data sources
Spark uses `DataFrameReader` (`spark.read`) and `DataFrameWriter` (`df.write`). Common built-in sources include CSV, JSON, Parquet, ORC and text. JDBC is supported for relational databases using an appropriate JDBC driver. Community connectors extend Spark to systems such as MongoDB, Elasticsearch and cloud services.

**Teaching note:** A source may require extra packages, drivers, permissions or a running server.

### 1. CSV: inference versus an explicit schema
CSV is interoperable but does not preserve column types. Compare inferred types with an explicit schema.

In [ ]:
csv_df = spark.read.option("header",True).option("inferSchema",True).csv(str(base / "sales.csv"))
csv_df.printSchema()
csv_df.show(5,truncate=False)

In [ ]:
schema = T.StructType([T.StructField("order_id",T.IntegerType()),T.StructField("order_date",T.StringType()),T.StructField("city",T.StringType()),T.StructField("category",T.StringType()),T.StructField("quantity",T.IntegerType()),T.StructField("unit_price",T.DoubleType()),T.StructField("channel",T.StringType())])
sales = spark.read.option("header",True).schema(schema).csv(str(base / "sales.csv"))
sales.printSchema()

### 2. JSON: semi-structured data
Spark reads JSON Lines by default (one JSON object per line). Nested JSON can become `struct` and `array` columns.

In [ ]:
json_df = spark.read.json(str(base / "sales.json"))
json_df.printSchema()
json_df.show(3,truncate=False)

### 3. Parquet: columnar storage
Parquet preserves schema and supports efficient column-oriented reads. Spark writes directories containing one or more part files, not a single `.parquet` file.

In [ ]:
parquet_path = str(base / "sales_parquet")
sales.write.mode("overwrite").parquet(parquet_path)
parquet_df = spark.read.parquet(parquet_path)
parquet_df.select("city","quantity").show()
print("Row count:",parquet_df.count())

### 4. Data source options, partitions, and save modes
Common writer modes: `errorifexists` (default), `append`, `overwrite`, and `ignore`. Partitioning can help queries when partition columns are frequently filtered, but over-partitioning creates small files.

In [ ]:
partition_path = str(base / "partitioned_sales")
sales.write.mode("overwrite").partitionBy("city").parquet(partition_path)
filtered = spark.read.parquet(partition_path).filter(F.col("city")=="Madrid")
filtered.explain(mode="formatted")
filtered.show()

### 5. JDBC (optional integration)
Spark can read PostgreSQL, MySQL, and other relational databases over JDBC. Requires a reachable database and a compatible JDBC driver on Spark’s classpath. **Do not put real passwords in a shared notebook.**

In [ ]:
# Illustrative only: do not execute without a database and JDBC driver.
# jdbc_df = (spark.read.format("jdbc")
#   .option("url", "jdbc:postgresql://localhost:5432/retail")
#   .option("dbtable", "public.orders")
#   .option("user", "YOUR_USER")
#   .option("password", "LOAD_FROM_SECRET_MANAGER")
#   .option("driver", "org.postgresql.Driver")
#   .load())

### 6. Community connector: MongoDB (20-minute lecture, 50–70 min)
MongoDB stores BSON documents; the Spark Connector translates between documents and Spark DataFrames. The connector version **must match your Spark and Scala versions**. For Spark 3.5 / Scala 2.12, a commonly used artifact is `org.mongodb.spark:mongo-spark-connector_2.12:10.4.1`. Verify compatibility for your deployment.

Example startup (in a terminal, not in an existing Spark session):
```bash
pyspark --packages org.mongodb.spark:mongo-spark-connector_2.12:10.4.1
```

You also need a MongoDB server and suitable access credentials. These cells are optional.

In [ ]:
# OPTIONAL — run only when MongoDB is available and the connector is installed
# mongo_uri = "mongodb://localhost:27017"
# mongo_df = (spark.read.format("mongodb")
#    .option("connection.uri",mongo_uri)
#    .option("database","teaching")
#    .option("collection","sales")
#    .load())
# mongo_df.printSchema()
# mongo_df.show(5)

In [ ]:
# OPTIONAL — writing Spark rows as MongoDB documents
# (sales.write.format("mongodb")
#  .mode("append")
#  .option("connection.uri", "mongodb://localhost:27017")
#  .option("database", "teaching")
#  .option("collection", "sales")
#  .save())

### Guided comparison
| Source | Schema | Typical use | Important caveat |
|---|---|---|---|
| CSV | External/inferred | Exchange | Types not preserved |
| JSON | Flexible/nested | Events/APIs | Inconsistent fields |
| Parquet | Embedded | Analytics/data lakes | Not human-readable |
| JDBC | Database-managed | SQL databases | Driver, credentials, partitioning |
| MongoDB | Document-oriented | Operational document data | Connector + server |

## 70–80 min · Challenge: Did you get it?
**Student tasks (8 min):** (1) Read the CSV with an explicit schema. (2) Write it as Parquet partitioned by `channel`. (3) Read the Parquet and count online orders. (4) Explain why Parquet may outperform CSV for analytical queries.

**Expected output:** online orders = **5**.

**Instructor discussion (2 min):** What changes if the input data doubles? What changes if the schema evolves?

In [ ]:
# TODO: Student challenge
# 1. Reuse sales from the lecture
# 2. Write partitioned Parquet to base / "challenge_parquet"
# 3. Read it back and count channel == "online"

### Instructor solution — reveal after the challenge

In [ ]:
challenge_path = str(base / "challenge_parquet")
sales.write.mode("overwrite").partitionBy("channel").parquet(challenge_path)
answer = spark.read.parquet(challenge_path).filter(F.col("channel")=="online").count()
assert answer == 5
print("Online orders:",answer)

## Recap / exit ticket
1. Why is an explicit schema useful for CSV?
2. Which file format preserves schema and is columnar?
3. What must be installed to connect Spark to MongoDB?
4. What does `mode("overwrite")` do?

**Next session:** schemas, rows, columns, expressions, conditions and aggregations.